In [ ]:
# ==============================================================================
# SCRIPT DE CONTRASTE DINÁMICO COMPLETAMENTE AUTOCONTENIDO (SPEI-6 vs. SPI-6)
# ==============================================================================

import pandas as pd
import numpy as np
import scipy.stats as stats
from linearmodels.panel import RandomEffects
from patsy import dmatrices

# 1. Cargar y preparar datos mensuales limpios
df = pd.read_csv('/home/hjvargaso/proyectos/tesis_sequia_2/data/processed/datos_mensuales_con_spei.csv')
df['fecha'] = pd.to_datetime(df['fecha'])
df['mes'] = df['fecha'].dt.month

# Mapeo de Clústeres (18 bases)
cluster_mapping = {
    'MINGA': 1, 'SJBA': 1, 'VILL': 1, 'CAAZ': 1, 'COVIE': 1, 'CMEZA': 1, 'ENCAR': 1,
    'MCAL': 2, 'SEST': 2, 'SPED': 2, 'GBRU': 2, 'CONC': 2, 'PCOL': 2, 'LUQUE': 2, 'PJC': 2, 'PTOC': 2, 'PILAR': 2, 'SGUA': 2
}
df['cluster'] = df['estacion_id'].map(cluster_mapping)


# 2. DEFINICIÓN ALGORÍTMICA DEL CÁLCULO DEL SPI-6 POR BASE
def calcular_spi_6_por_estacion(df_datos):
    # Asegurar orden cronológico
    df_datos = df_datos.sort_values(by=['estacion_id', 'fecha']).reset_index(drop=True)
    
    # Calcular suma móvil de 6 meses de precipitación
    df_datos['lluvia_6'] = df_datos.groupby('estacion_id')['lluvia_total'].transform(
        lambda x: x.rolling(6, min_periods=6).sum()
    )
    
    spi_global = pd.Series(index=df_datos.index, dtype=float)
    
    # Bucle por estación y por mes para remover estacionalidad
    for est in df_datos['estacion_id'].unique():
        idx_est = df_datos['estacion_id'] == est
        for m in range(1, 13):
            idx_mes_est = idx_est & (df_datos['mes'] == m)
            datos_mes = df_datos.loc[idx_mes_est, 'lluvia_6'].dropna()
            
            if len(datos_mes) < 10: # Mínimo de datos para ajustar
                continue
                
            # Probabilidad de precipitación nula (q)
            zeros = (datos_mes == 0).sum()
            q = zeros / len(datos_mes)
            
            non_zero_data = datos_mes[datos_mes > 0]
            
            if len(non_zero_data) > 3:
                # Ajustar distribución Gamma
                fit_alpha, fit_loc, fit_beta = stats.gamma.fit(non_zero_data, floc=0)
                cdf_non_zero = stats.gamma.cdf(non_zero_data, fit_alpha, fit_loc, fit_beta)
            else:
                cdf_non_zero = non_zero_data * 0.0
                
            # Reconstruir CDF (H(x))
            cdf_all = pd.Series(index=datos_mes.index, dtype=float)
            cdf_all[datos_mes == 0] = q
            cdf_all[datos_mes > 0] = q + (1 - q) * cdf_non_zero
            
            # Acotar probabilidades para evitar infinitos en ppf
            cdf_all = np.clip(cdf_all, 0.0001, 0.9999)
            
            # Normalización (Z-score)
            spi_global.loc[datos_mes.index] = stats.norm.ppf(cdf_all)
            
    return spi_global


# Ejecutar el cálculo del SPI-6
df['spi_6'] = calcular_spi_6_por_estacion(df)

# Calcular la diferencia directa entre índices (Efecto Térmico aislado)
df['diff_6'] = df['spei_6'] - df['spi_6']


# 3. ESTIMACIÓN DE LOS MODELOS PARALELOS CON ERRORES DE DRISCOLL-KRAAY
df['tiempo'] = (df['fecha'].dt.year - 2000) * 12 + df['fecha'].dt.month
df['estacion_climatica'] = np.where(df['fecha'].dt.month.isin([12, 1, 2]), 'verano',
                           np.where(df['fecha'].dt.month.isin([3, 4, 5]), 'otono',
                           np.where(df['fecha'].dt.month.isin([9, 10, 11]), 'primavera', 'invierno')))\

df['region_geografica'] = df['cluster'].astype(str)

df_panel = df.set_index(['estacion_id', 'fecha']).dropna(subset=['spei_6', 'spi_6', 'diff_6'])

# Matrices de diseño comunes
formula_spei = "spei_6 ~ 1 + tiempo * C(estacion_climatica, Treatment('invierno')) * C(region_geografica, Treatment('2'))"
formula_spi  = "spi_6 ~ 1 + tiempo * C(estacion_climatica, Treatment('invierno')) * C(region_geografica, Treatment('2'))"
formula_diff = "diff_6 ~ 1 + tiempo * C(estacion_climatica, Treatment('invierno')) * C(region_geografica, Treatment('2'))"

y_spei, X = dmatrices(formula_spei, df_panel, return_type='dataframe')
y_spi, _  = dmatrices(formula_spi, df_panel, return_type='dataframe')
y_diff, _ = dmatrices(formula_diff, df_panel, return_type='dataframe')

# Estimación con Driscoll-Kraay (Panel HAC)
re_spei = RandomEffects(y_spei, X).fit(cov_type='kernel', kernel='bartlett')
re_spi  = RandomEffects(y_spi, X).fit(cov_type='kernel', kernel='bartlett')
re_diff = RandomEffects(y_diff, X).fit(cov_type='kernel', kernel='bartlett')


# 4. FUNCIÓN AUXILIAR PARA LA EVALUACIÓN DINÁMICA DE LA SIGNIFICANCIA
def get_linear_combination_stats(model, vars_list):
    coefs = model.params
    cov_matrix = model.cov
    
    # Vector de pesos dinámico buscando por nombre de variable
    w = np.zeros(len(coefs))
    for var in vars_list:
        if var in coefs.index:
            w[coefs.index.get_loc(var)] = 1.0
            
    slope = np.dot(w, coefs)
    var_slope = np.dot(np.dot(w, cov_matrix), w)
    se = np.sqrt(var_slope) if var_slope > 0 else np.nan
    t_stat = slope / se if se > 0 else np.nan
    p_val = 2 * (1 - stats.norm.cdf(abs(t_stat))) if se > 0 else np.nan
    
    return slope, p_val

def get_stars(p):
    if p < 0.01: return '^{***}'
    elif p < 0.05: return '^{**}'
    elif p < 0.1: return '^{*}'
    return ''


# 5. GENERACIÓN DINÁMICA DE LA TABLA EN LATEX
t_base = 'tiempo'
t_otono = "tiempo:C(estacion_climatica, Treatment('invierno'))[T.otono]"
t_prim  = "tiempo:C(estacion_climatica, Treatment('invierno'))[T.primavera]"
t_ver   = "tiempo:C(estacion_climatica, Treatment('invierno'))[T.verano]"
t_reg1  = "tiempo:C(region_geografica, Treatment('2'))[T.1]"
t_otono_reg1 = "tiempo:C(estacion_climatica, Treatment('invierno'))[T.otono]:C(region_geografica, Treatment('2'))[T.1]"
t_prim_reg1  = "tiempo:C(estacion_climatica, Treatment('invierno'))[T.primavera]:C(region_geografica, Treatment('2'))[T.1]"
t_ver_reg1   = "tiempo:C(estacion_climatica, Treatment('invierno'))[T.verano]:C(region_geografica, Treatment('2'))[T.1]"

scenarios_trends = {
    'C1_verano':    {'region': 'Subregión sur-este (Clúster 1)',     'estacion': 'Verano',    'vars': [t_base, t_ver, t_reg1, t_ver_reg1]},
    'C1_otono':     {'region': 'Subregión sur-este (Clúster 1)',     'estacion': 'Otoño',     'vars': [t_base, t_otono, t_reg1, t_otono_reg1]},
    'C2_primavera': {'region': 'Subregión centro-chaco (Clúster 2)', 'estacion': 'Primavera', 'vars': [t_base, t_prim]},
    'C2_verano':    {'region': 'Subregión centro-chaco (Clúster 2)', 'estacion': 'Verano',    'vars': [t_base, t_ver]}
}

print("\n% ========================================================")
print("% CUADRO 4.5: CONTRASTE DE MODELOS EN LATEX (EVALUACIÓN REAL)")
print("% ========================================================")
print("\\begin{table}[ht!]")
print("\\centering")
print("\\caption{Comparación empírica de tendencias anuales netas y significancia del efecto del calentamiento global (SPEI-6 vs. SPI-6).}")
print("\\label{tab:contraste_spei_spi}")
print("\\resizebox{\\textwidth}{!}{%")
print("\\begin{tabular}{llcccl}")
print("\\hline")
print("\\textbf{Región Climática} & \\textbf{Estación del Año} & \\textbf{Tendencia SPEI-6} & \\textbf{Tendencia SPI-6} & \\textbf{Diferencia ($\\Delta \\beta$)} & \\textbf{Efecto del Calentamiento Global} \\\\ \\hline")

for sc_name, sc_info in scenarios_trends.items():
    # Estimar estadísticos reales para cada uno de los tres modelos
    slope_spei, p_spei = get_linear_combination_stats(re_spei, sc_info['vars'])
    slope_spi, p_spi   = get_linear_combination_stats(re_spi, sc_info['vars'])
    slope_diff, p_diff = get_linear_combination_stats(re_diff, sc_info['vars'])
    
    # Anualizar las tendencias netas para la tabla
    slope_spei_anual = slope_spei * 12
    slope_spi_anual  = slope_spi * 12
    slope_diff_anual = slope_diff * 12
    
    stars_spei = get_stars(p_spei)
    stars_spi  = get_stars(p_spi)
    stars_diff = get_stars(p_diff)
    
    # Decidir etiqueta del efecto basada en el p-valor real de la diferencia
    if p_diff < 0.05:
        effect_label = "Significativo (Aridificación inducida por ETP)."
    elif p_diff < 0.10:
        effect_label = "Sugerente (Tendencia de aridificación moderada)."
    else:
        effect_label = "No significativo (Precipitación domina el balance)."
        
    print(f"{sc_info['region']} & {sc_info['estacion']} & {slope_spei_anual:.4f}{stars_spei} & {slope_spi_anual:.4f}{stars_spi} & {slope_diff_anual:.4f}{stars_diff} & {effect_label} \\\\")

print("\\hline")
print("\\multicolumn{6}{l}{\\small Nota: *** $p < 0.01$, ** $p < 0.05$, * $p < 0.1$. Los valores corresponden a las tendencias anualizadas estimadas.} \\\\")
print("\\multicolumn{6}{l}{\\small Errores estándar y p-valores calculados de forma dinámica mediante la matriz de covarianza de Driscoll-Kraay.} \\\\")
print("\\end{tabular}%")
print("}")
print("\\end{table}\n")


% ========================================================
% CUADRO 4.5: CONTRASTE DE MODELOS EN LATEX (EVALUACIÓN REAL)
% ========================================================
\begin{table}[ht!]
\centering
\caption{Comparación empírica de tendencias anuales netas y significancia del efecto del calentamiento global (SPEI-6 vs. SPI-6).}
\label{tab:contraste_spei_spi}
\resizebox{\textwidth}{!}{%
\begin{tabular}{llcccl}
\hline
\textbf{Región Climática} & \textbf{Estación del Año} & \textbf{Tendencia SPEI-6} & \textbf{Tendencia SPI-6} & \textbf{Diferencia ($\Delta \beta$)} & \textbf{Efecto del Calentamiento Global} \\ \hline
Subregión sur-este (Clúster 1) & Verano & -0.0400^{*} & -0.0339 & -0.0060^{*} & Sugerente (Tendencia de aridificación moderada). \\
Subregión sur-este (Clúster 1) & Otoño & -0.0086 & -0.0057 & -0.0029 & No significativo (Precipitación domina el balance). \\
Subregión centro-chaco (Clúster 2) & Primavera & -0.0161 & -0.0059 & -0.0102^{*} & Sugerente (Tendencia de ar